# Pricing Data Validation

This notebook validates the completeness of pricing data by checking:
1. **DBU Rates** (`instance_rates`) - Do we have DBU rates for all instance types?
2. **VM Costs** (`vm_costs`) - Do we have VM costs (on-demand, spot, reserved) for all instance types?
3. **Join Coverage** - Which instance types have both DBU rates AND VM costs?


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_INSTANCE_RATES = f"{CATALOG}.{SCHEMA}.instance_rates"
TABLE_VM_COSTS = f"{CATALOG}.{SCHEMA}.vm_costs"

print(f"✅ Validating: {CATALOG}.{SCHEMA}")


In [ ]:
# 1. Overview of both tables
print("=" * 60)
print("📊 TABLE OVERVIEW")
print("=" * 60)

# Instance rates summary
print("\n📋 INSTANCE_RATES (DBU rates):")
display(spark.sql(f"""
    SELECT cloud, COUNT(DISTINCT instance_type) as instance_types, COUNT(*) as total_rows
    FROM {TABLE_INSTANCE_RATES}
    GROUP BY cloud
    ORDER BY cloud
"""))

# VM costs summary
print("\n📋 VM_COSTS (VM pricing):")
display(spark.sql(f"""
    SELECT cloud, pricing_tier, COUNT(DISTINCT instance_type) as instance_types, COUNT(DISTINCT region) as regions
    FROM {TABLE_VM_COSTS}
    GROUP BY cloud, pricing_tier
    ORDER BY cloud, pricing_tier
"""))


In [ ]:
# 2. Coverage Analysis - Which instance types have VM costs?
print("=" * 60)
print("📊 COVERAGE ANALYSIS BY CLOUD")
print("=" * 60)

for cloud in ['AWS', 'AZURE', 'GCP']:
    print(f"\n{'='*40}")
    print(f"☁️  {cloud}")
    print(f"{'='*40}")
    
    display(spark.sql(f"""
        WITH dbu_instances AS (
            SELECT DISTINCT instance_type FROM {TABLE_INSTANCE_RATES} WHERE cloud = '{cloud}'
        ),
        vm_coverage AS (
            SELECT 
                instance_type,
                MAX(CASE WHEN pricing_tier = 'on_demand' THEN 1 ELSE 0 END) as has_on_demand,
                MAX(CASE WHEN pricing_tier = 'spot' THEN 1 ELSE 0 END) as has_spot,
                MAX(CASE WHEN pricing_tier = 'reserved_1y' THEN 1 ELSE 0 END) as has_reserved_1y,
                MAX(CASE WHEN pricing_tier = 'reserved_3y' THEN 1 ELSE 0 END) as has_reserved_3y
            FROM {TABLE_VM_COSTS}
            WHERE cloud = '{cloud}'
            GROUP BY instance_type
        )
        SELECT 
            'Total DBU instances' as metric,
            COUNT(*) as count
        FROM dbu_instances
        UNION ALL
        SELECT 
            'Has On-Demand' as metric,
            COUNT(*) as count
        FROM dbu_instances d
        JOIN vm_coverage v ON d.instance_type = v.instance_type
        WHERE v.has_on_demand = 1
        UNION ALL
        SELECT 
            'Has Spot' as metric,
            COUNT(*) as count
        FROM dbu_instances d
        JOIN vm_coverage v ON d.instance_type = v.instance_type
        WHERE v.has_spot = 1
        UNION ALL
        SELECT 
            'Has Reserved 1Y' as metric,
            COUNT(*) as count
        FROM dbu_instances d
        JOIN vm_coverage v ON d.instance_type = v.instance_type
        WHERE v.has_reserved_1y = 1
        UNION ALL
        SELECT 
            'Has Reserved 3Y' as metric,
            COUNT(*) as count
        FROM dbu_instances d
        JOIN vm_coverage v ON d.instance_type = v.instance_type
        WHERE v.has_reserved_3y = 1
        UNION ALL
        SELECT 
            'Has ALL tiers' as metric,
            COUNT(*) as count
        FROM dbu_instances d
        JOIN vm_coverage v ON d.instance_type = v.instance_type
        WHERE v.has_on_demand = 1 AND v.has_spot = 1 AND v.has_reserved_1y = 1 AND v.has_reserved_3y = 1
    """))


In [ ]:
# 3. Missing Instance Types - DBU rates without VM costs
print("=" * 60)
print("❌ MISSING VM COSTS (have DBU rates but no VM pricing)")
print("=" * 60)

for cloud in ['AWS', 'AZURE', 'GCP']:
    print(f"\n☁️  {cloud} - Missing On-Demand:")
    display(spark.sql(f"""
        SELECT d.instance_type
        FROM {TABLE_INSTANCE_RATES} d
        LEFT JOIN (
            SELECT DISTINCT instance_type 
            FROM {TABLE_VM_COSTS} 
            WHERE cloud = '{cloud}' AND pricing_tier = 'on_demand'
        ) v ON d.instance_type = v.instance_type
        WHERE d.cloud = '{cloud}' AND v.instance_type IS NULL
        ORDER BY d.instance_type
        LIMIT 20
    """))


In [ ]:
# 4. Coverage Summary Table
print("=" * 60)
print("📊 COVERAGE SUMMARY (% of DBU instances with VM pricing)")
print("=" * 60)

display(spark.sql(f"""
    WITH dbu_counts AS (
        SELECT cloud, COUNT(DISTINCT instance_type) as total_dbu
        FROM {TABLE_INSTANCE_RATES}
        GROUP BY cloud
    ),
    vm_counts AS (
        SELECT 
            i.cloud,
            COUNT(DISTINCT CASE WHEN v.pricing_tier = 'on_demand' THEN i.instance_type END) as on_demand,
            COUNT(DISTINCT CASE WHEN v.pricing_tier = 'spot' THEN i.instance_type END) as spot,
            COUNT(DISTINCT CASE WHEN v.pricing_tier = 'reserved_1y' THEN i.instance_type END) as reserved_1y,
            COUNT(DISTINCT CASE WHEN v.pricing_tier = 'reserved_3y' THEN i.instance_type END) as reserved_3y
        FROM {TABLE_INSTANCE_RATES} i
        LEFT JOIN {TABLE_VM_COSTS} v ON i.instance_type = v.instance_type AND i.cloud = v.cloud
        GROUP BY i.cloud
    )
    SELECT 
        d.cloud,
        d.total_dbu as total_instances,
        v.on_demand,
        ROUND(100.0 * v.on_demand / d.total_dbu, 1) as on_demand_pct,
        v.spot,
        ROUND(100.0 * v.spot / d.total_dbu, 1) as spot_pct,
        v.reserved_1y,
        ROUND(100.0 * v.reserved_1y / d.total_dbu, 1) as reserved_1y_pct,
        v.reserved_3y,
        ROUND(100.0 * v.reserved_3y / d.total_dbu, 1) as reserved_3y_pct
    FROM dbu_counts d
    JOIN vm_counts v ON d.cloud = v.cloud
    ORDER BY d.cloud
"""))


In [ ]:
# 5. Sample Join - Show combined pricing data
print("=" * 60)
print("✅ SAMPLE JOINED DATA (DBU rate + VM cost)")
print("=" * 60)

display(spark.sql(f"""
    SELECT 
        i.cloud,
        i.instance_type,
        i.instance_family,
        i.vcpus,
        i.memory_gb,
        i.dbu_rate,
        v.pricing_tier,
        v.cost_per_hour as vm_cost_per_hour,
        ROUND(i.dbu_rate + v.cost_per_hour, 4) as total_cost_per_hour
    FROM {TABLE_INSTANCE_RATES} i
    JOIN {TABLE_VM_COSTS} v 
        ON i.instance_type = v.instance_type 
        AND i.cloud = v.cloud
    WHERE v.pricing_tier = 'on_demand'
    ORDER BY i.cloud, i.instance_type
    LIMIT 30
"""))


In [ ]:
# 6. Region Coverage Check
print("=" * 60)
print("📍 REGION COVERAGE (VM costs per region)")
print("=" * 60)

display(spark.sql(f"""
    SELECT 
        cloud,
        region,
        COUNT(DISTINCT instance_type) as instance_types,
        SUM(CASE WHEN pricing_tier = 'on_demand' THEN 1 ELSE 0 END) as on_demand_count,
        SUM(CASE WHEN pricing_tier = 'spot' THEN 1 ELSE 0 END) as spot_count,
        SUM(CASE WHEN pricing_tier = 'reserved_1y' THEN 1 ELSE 0 END) as reserved_1y_count,
        SUM(CASE WHEN pricing_tier = 'reserved_3y' THEN 1 ELSE 0 END) as reserved_3y_count
    FROM {TABLE_VM_COSTS}
    GROUP BY cloud, region
    ORDER BY cloud, region
"""))
